# Feature extraction for the synthetic fashion SKU demand dataset.

In [1]:
# libraries
import pandas as pd
import numpy as np
import re

# for registering as Vertex AI Tabular Dataset
from google.cloud import bigquery, storage
from google.cloud import aiplatform

# import params
import functions.params_utils

### params

In [4]:
params = functions.params_utils.load_params()

In [5]:
GRANULARITY = params["features"]["granularity"] # "W"

FORECAST_HORIZON = params["features"]["forecast_horizon"]       # 8 weeks to predect
VALIDATION_HORIZON = params["features"]["validation_horizon"]   # 104 weeks held out for validation before test

# setting lags to 0, 1, 2, 4, 44 for weekly data (or 0, 6, 13, 29, 364 for daily)
_LAG_OFFSETS = [0, 1, 2, 4, 44] if GRANULARITY == "W" else [0, 6, 13, 29, 364]
LAGS = [FORECAST_HORIZON + off for off in _LAG_OFFSETS]  # e.g. [8, 9, 10, 12, 52]

# setting rolling windows to 4, 12 for weekly data (or 7, 30 for daily)
ROLLING_WINDOWS = [4, 12] if GRANULARITY == "W" else [7, 30]

In [6]:
project = !gcloud config get-value project
PROJECT_ID = project[0]
print(f"Active Project ID: {PROJECT_ID}")

REGION = "us-central1"

# Clients
bq = bigquery.Client(project=PROJECT_ID)
gcs = storage.Client(project=PROJECT_ID)

# GCS Parameters & Directory Path
BUCKET = PROJECT_ID

Active Project ID: fashionmvforecast


### clear old data if exists

In [5]:
def delete_gcs_file_if_exists(bucket_name: str, blob_name: str):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    if blob.exists():
        blob.delete()
        print(f"Deleted gs://{bucket_name}/{blob_name}")
    else:
        print(f"File gs://{bucket_name}/{blob_name} does not exist. Skipping.")

delete_gcs_file_if_exists(BUCKET, "data/input/synthetic_fashion_demand_features.csv")
delete_gcs_file_if_exists(BUCKET, "data/input/train_features.csv")
delete_gcs_file_if_exists(BUCKET, "data/input/val_features.csv")
delete_gcs_file_if_exists(BUCKET, "data/input/test_features.csv")

File gs://fashionmvforecast/data/input/synthetic_fashion_demand_features.csv does not exist. Skipping.
File gs://fashionmvforecast/data/input/train_features.csv does not exist. Skipping.
File gs://fashionmvforecast/data/input/val_features.csv does not exist. Skipping.
File gs://fashionmvforecast/data/input/test_features.csv does not exist. Skipping.


### Read data

In [9]:
GCS_DIR = "gs://fashionmvforecast/data/input"
df = pd.read_csv(f"{GCS_DIR}/synthetic_fashion_demand.csv", parse_dates=["date"])

### Call feature functions in folder "functions"

In [7]:
from functions.Features_Functions import get_train_cutoff, add_abc_classification, add_xyz_classification, add_calendar_features, add_lifecycle_features, add_price_features, add_target_lag_features, add_endogenous_lagged_features, add_cross_product_features, encode_categoricals, time_based_split
from functions.Features_Functions import (
    TIME_COLUMN, SERIES_ID_COLUMN, TARGET_COLUMN,
    AVAILABLE_AT_FORECAST, UNAVAILABLE_AT_FORECAST,
)


In [8]:
train_cutoff = get_train_cutoff(df)
print(f"train cutoff for ABC/XYZ classification: {train_cutoff.date()}")

df = add_abc_classification(df, as_of=train_cutoff)
df = add_xyz_classification(df, as_of=train_cutoff)
df = add_calendar_features(df)
df = add_lifecycle_features(df)
df = add_price_features(df)
df = add_target_lag_features(df)
df = add_endogenous_lagged_features(df)
df = add_cross_product_features(df)
df = encode_categoricals(df)
df, train, val, test = time_based_split(df)


train cutoff for ABC/XYZ classification: 2022-12-25


In [9]:
print(list(df.columns))

['sku_id', 'date', 'category', 'color', 'material', 'brand', 'size_range', 'price', 'discount_pct', 'promo_flag', 'markdown_stage', 'competitor_price_index', 'week_of_year', 'month', 'quarter', 'is_holiday', 'is_weekend', 'fashion_week_flag', 'back_to_school_flag', 'payday_week_flag', 'days_since_launch', 'temperature', 'precipitation', 'marketing_spend', 'campaign_flag', 'lead_time_days', 'store_count', 'channel', 'cannibalization_index', 'substitute_price_ratio', 'sales_qty', 'abc_class', 'xyz_class', 'woy_sin', 'woy_cos', 'month_sin', 'month_cos', 'lifecycle_stage', 'price_change_pct', 'relative_price_vs_category', 'had_promo', 'weeks_since_last_promo', 'sales_qty_lag_8', 'sales_qty_lag_9', 'sales_qty_lag_10', 'sales_qty_lag_12', 'sales_qty_lag_52', 'sales_qty_rollmean_4', 'sales_qty_rollstd_4', 'sales_qty_rollmean_12', 'sales_qty_rollstd_12', 'review_count_lag8', 'avg_rating_lag8', 'wishlist_adds_lag8', 'social_trend_score_lag8', 'inventory_on_hand_lag8', 'stockout_flag_lag8', 'sub

### Replaces spaces/special chars with underscores and strips leading/trailing underscores

In [10]:
df.columns = [re.sub(r'[\W]+', '_', col).strip('_') for col in df.columns]
df.columns = [re.sub(r'[\W]+', '_', col).strip('_') for col in df.columns]

In [11]:
print(list(df.columns))

['sku_id', 'date', 'category', 'color', 'material', 'brand', 'size_range', 'price', 'discount_pct', 'promo_flag', 'markdown_stage', 'competitor_price_index', 'week_of_year', 'month', 'quarter', 'is_holiday', 'is_weekend', 'fashion_week_flag', 'back_to_school_flag', 'payday_week_flag', 'days_since_launch', 'temperature', 'precipitation', 'marketing_spend', 'campaign_flag', 'lead_time_days', 'store_count', 'channel', 'cannibalization_index', 'substitute_price_ratio', 'sales_qty', 'abc_class', 'xyz_class', 'woy_sin', 'woy_cos', 'month_sin', 'month_cos', 'lifecycle_stage', 'price_change_pct', 'relative_price_vs_category', 'had_promo', 'weeks_since_last_promo', 'sales_qty_lag_8', 'sales_qty_lag_9', 'sales_qty_lag_10', 'sales_qty_lag_12', 'sales_qty_lag_52', 'sales_qty_rollmean_4', 'sales_qty_rollstd_4', 'sales_qty_rollmean_12', 'sales_qty_rollstd_12', 'review_count_lag8', 'avg_rating_lag8', 'wishlist_adds_lag8', 'social_trend_score_lag8', 'inventory_on_hand_lag8', 'stockout_flag_lag8', 'sub

In [12]:
df.splits.unique()

<ArrowStringArray>
['TRAIN', 'VALIDATE', 'TEST']
Length: 3, dtype: str

### Create future data (for prediction)

In [ ]:
# TIME_COLUMN / SERIES_ID_COLUMN / TARGET_COLUMN / AVAILABLE_AT_FORECAST /
# UNAVAILABLE_AT_FORECAST all come from functions.Features_Functions (imported
# above) 
# FORECAST_HORIZON itself already comes from params.yaml (loaded earlier in
# this notebook)

freq = pd.infer_freq(sorted(df[TIME_COLUMN].unique())[:10]) or "7D"  # weekly data -> "7D"

# Calendar week-number sets, matching 1_Generate_Synthetic_Data.ipynb's
# make_calendar() exactly. These have to be *recomputed* for the future
# dates, not carried forward -- they depend on which week a future date
# falls in, not on anything known at the SKU's last observed row.
_HOLIDAY_WEEKS = {1, 7, 20, 47, 48, 51, 52}
_FASHION_WEEKS = {7, 8, 9, 20, 21, 22, 36, 37, 38, 47, 48, 49}
_BACK_TO_SCHOOL_WEEKS = {33, 34, 35}

future_rows = []
for sku_id, group in df.groupby(SERIES_ID_COLUMN):
    last_row = group.sort_values(TIME_COLUMN).iloc[-1]
    last_date = last_row[TIME_COLUMN]
    future_dates = pd.date_range(start=last_date, periods=FORECAST_HORIZON + 1, freq=freq)[1:]

    for step, d in enumerate(future_dates, start=1):
        row = {c: np.nan for c in df.columns}

        # Frozen carry-forward: every AVAILABLE_AT_FORECAST column this dataset
        # has no forward-looking plan for (price, promo, marketing spend, store
        # count, category/brand/size attributes, lifecycle_stage, ...) is fixed
        # at the SKU's last known value. "Available at forecast" here means
        # "frozen at its last known value", not "known in advance" -- honest
        # given we have no real pricing/promo calendar -- but it's still a real
        # value, which is what Vertex AI requires; a null here is exactly the
        # "Missing struct property: <column>" rejection this fixes.
        for c in AVAILABLE_AT_FORECAST:
            if c in last_row.index:
                row[c] = last_row[c]

        # sku_id / date identify this row and must reflect the future row being
        # built, not the SKU's last observed row -- set these *after* the
        # AVAILABLE_AT_FORECAST carry-forward above (TIME_COLUMN is itself one of
        # those columns, so setting it before that loop would get silently
        # overwritten right back to last_date).
        row[SERIES_ID_COLUMN] = sku_id
        row[TIME_COLUMN] = d

        # Deterministic, date-derived columns: recompute for real, overriding
        # the frozen carry-forward above.
        week_of_year = d.isocalendar()[1]
        row["week_of_year"] = week_of_year
        row["month"], row["quarter"] = d.month, d.quarter
        row["is_weekend"] = int(d.dayofweek in (5, 6))
        row["is_holiday"] = week_of_year in _HOLIDAY_WEEKS
        row["fashion_week_flag"] = week_of_year in _FASHION_WEEKS
        row["back_to_school_flag"] = week_of_year in _BACK_TO_SCHOOL_WEEKS
        row["payday_week_flag"] = d.day <= 7
        row["woy_sin"] = np.sin(2 * np.pi * week_of_year / 52)
        row["woy_cos"] = np.cos(2 * np.pi * week_of_year / 52)
        row["month_sin"] = np.sin(2 * np.pi * row["month"] / 12)
        row["month_cos"] = np.cos(2 * np.pi * row["month"] / 12)

        # days_since_launch keeps advancing by 7 days/week regardless of the
        # frozen lifecycle_stage above -- recomputing lifecycle_stage here from
        # this future days_since_launch would risk it disagreeing with the
        # (also carried-forward) lifecycle_stage_copy_* one-hot columns, so
        # both stay frozen together instead.
        row["days_since_launch"] = last_row["days_since_launch"] + 7 * step

        # UNAVAILABLE_AT_FORECAST columns (the target included) stay null --
        # already the dict's default above; set explicitly so the intent is
        # visible here and survives even if a column's default ever changes.
        for c in UNAVAILABLE_AT_FORECAST:
            row[c] = np.nan

        future_rows.append(row)

future_df = pd.DataFrame(future_rows)[list(df.columns)]


### Write to folder "gs://fashionmvforecast/data/input" again

In [13]:
GCS_DIR = "gs://fashionmvforecast/data/input"

df.to_csv(f"{GCS_DIR}/synthetic_fashion_demand_features.csv", index=False)
train.to_csv(f"{GCS_DIR}/train_features.csv", index=False)
val.to_csv(f"{GCS_DIR}/val_features.csv", index=False)
test.to_csv(f"{GCS_DIR}/test_features.csv", index=False)
future_df.to_csv(f"{GCS_DIR}/future_df.csv", index=False)

print("All datasets successfully written to GCS!")

All datasets successfully written to GCS!
